# 第 1 章 §1.3 —— Prompt Chaining 逐行拆解

> 这是书里 `Chapter_01_Prompt_Chaining_(Code_Example)` 的**注释版**。
> 代码与书完全一致，只做了两处非功能性改动：
> ① 补了 `model="deepseek-flash"`（本书默认 `gpt-3.5-turbo`，DeepSeek 不认）；
> ② 加了中文注释。
>
> **怎么读**：先看下面带注释的代码，再读「执行时序」和「六个关键点」。
> 想动手调试就用同目录下的 `../../ch01_prompt_chaining/01_3_debug_lab.ipynb`（已拆成 9 格）。

## 一句话概括这个脚本

> 把「从一段自然语言里抽出规格」和「把规格转成 JSON」拆成两次独立的
> LLM 调用，中间用**一个 dict 的键名**把两段接起来。

它做的**不是**"让模型一次输出 JSON"，而是"让模型分两次、每次只干一件事"。
这正是 Prompt Chaining 的全部主张。

In [1]:
# =============================================================================
# 第 1 章 §1.3 完整代码（逐行注释）
# =============================================================================
#
# 全脚本可以分成两个阶段：
#   阶段 A（第 ①②③④⑤⑥ 步）：只是【搭积木】，一行网络请求都不会发
#   阶段 B（第 ⑦ 步）        ：invoke() 点火，才真正开始调用模型
#
# 这个区分是理解 LangChain 的第一道门槛，也是最容易误解的地方。

# os 在这里其实【没被使用】。书里保留它，是为上面注释掉的 load_dotenv() 铺垫。
import os

# 三个 import 对应链上三种截然不同的角色：
#   ChatPromptTemplate  —— 纯本地字符串插值，把变量填进模板（不花 token）
#   ChatOpenAI          —— 真正调用模型的那一环（唯一花 token 的）
#   StrOutputParser     —— 纯本地类型转换，把 AIMessage 拆成字符串（不花 token）
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ─────────────────────────────────────────────────────────────────────────────
# ① 创建模型客户端
# ─────────────────────────────────────────────────────────────────────────────
# 注意：这一步【不发任何网络请求】。它只是构造一个客户端对象，
# 内部会准备好 httpx.Client、鉴权头、base_url 等。
# （所以之前那个代理 scheme 报错，就是在这行炸的，而不是在 invoke 时。）
#
# temperature=0：让采样尽量取概率最高的 token。降低随机性，但【不等于确定】。
# model="deepseek-flash"：书里没有。ChatOpenAI 默认模型硬编码为 "gpt-3.5-turbo"，
# 而 DeepSeek 只认 deepseek-flash / deepseek-v4-pro。
llm = ChatOpenAI(temperature=0, model="deepseek-flash")


# ─────────────────────────────────────────────────────────────────────────────
# ② 第一个 prompt 模板：负责「抽取」
# ─────────────────────────────────────────────────────────────────────────────
# from_template 会扫描字符串里的 {xxx}，把它登记为【输入变量】。
# 这里只有 {text_input} 一个变量，所以之后必须传 {"text_input": ...}，
# 少传会抛 KeyError，多传会被静默忽略（容易埋坑）。
prompt_extract = ChatPromptTemplate.from_template(
    "Extract the technical specifications from the following text:\n\n{text_input}"
)


# ─────────────────────────────────────────────────────────────────────────────
# ③ 第二个 prompt 模板：负责「转成 JSON」
# ─────────────────────────────────────────────────────────────────────────────
# 它的输入变量叫 {specifications}。这个名字不是随便起的 ——
# 它必须和第 ⑤ 步那个 dict 的键名【完全一致】，否则运行时报 KeyError。
# 这就是两段之间的「接口契约」，靠字符串名字维系。
#
# 一个有意思的细节：这里【没有】写 JSON 字面量示例（比如 '{"cpu": ...}'）。
# 因为 ChatPromptTemplate 会把花括号当变量占位符，
# 想输出字面量大括号必须写成 {{ 和 }} 转义。
# 作者改用自然语言描述字段名，恰好绕开了这个坑 ——
# 这是规避，不是解决。后面章节讲 structured output 才是正解。
prompt_transform = ChatPromptTemplate.from_template(
    "Transform the following specifications into a JSON object with 'cpu', 'memory', and 'storage' as keys:\n\n{specifications}"
)


# ─────────────────────────────────────────────────────────────────────────────
# ④ 组装第一个链（extraction_chain）
# ─────────────────────────────────────────────────────────────────────────────
# 这里的 | 是 Python 的按位或运算符，被 LangChain 重载成了「串联」。
# 等价写法：RunnableSequence([prompt_extract, llm, StrOutputParser()])
#
# 【重点】这一行【不执行任何东西】。它只是构造出一个 Runnable 对象。
# 同理：整条 full_chain 组装完，一次模型都还没调用。
extraction_chain = prompt_extract | llm | StrOutputParser()


# ─────────────────────────────────────────────────────────────────────────────
# ⑤ 组装完整链（全篇最精妙，也最脆弱的一行）
# ─────────────────────────────────────────────────────────────────────────────
# {"specifications": extraction_chain} 是个 dict。LangChain 会把
# 「只有一个键的 dict」自动转成 RunnableParallel，语义是：
#
#   1. 收到上游输入 {"text_input": "..."}
#   2. 把这个输入【原封不动】地喂给 extraction_chain
#   3. 把它的返回值，装进键名 "specifications"
#   4. 输出 {"specifications": "<抽出来的文本>"}
#
# 于是键名把两段粘在了一起：
#   extraction_chain 的输出  →  prompt_transform 的 {specifications} 变量
#
# ⚠️ 这个契约【完全不受校验保护】：
#   - 键名写错 → 运行时报 KeyError（还算好的，至少报错）
#   - 内容错了 → 无人察觉，直接往下传（这才是真正的危险）
full_chain = (
    {"specifications": extraction_chain}
    | prompt_transform
    | llm
    | StrOutputParser()
)


# ─────────────────────────────────────────────────────────────────────────────
# ⑥ 准备输入
# ─────────────────────────────────────────────────────────────────────────────
input_text = "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD."


# ─────────────────────────────────────────────────────────────────────────────
# ⑦ 点火：真正开始执行
# ─────────────────────────────────────────────────────────────────────────────
# invoke() 才是「执行」的开始。这一行内部发生的事（见下一个 markdown 格）：
#
#   第 1 次 API 调用：
#     prompt_extract 插值 → llm 抽取规格 → StrOutputParser 转字符串
#   dict 打包：{"specifications": "..."}
#   第 2 次 API 调用：
#     prompt_transform 插值 → llm 转 JSON → StrOutputParser 转字符串
#
# 两次调用【严格串行】：第二次的输入依赖第一次的输出。
# 这就是 chaining 的本质 —— 也是它的代价（延迟叠加、错误累积）。
final_result = full_chain.invoke({"text_input": input_text})


# ─────────────────────────────────────────────────────────────────────────────
# ⑧ 打印
# ─────────────────────────────────────────────────────────────────────────────
# ⚠️ 关键认知：final_result 是【字符串】，不是 dict。
# 它只是「长得像 JSON」。这里：
#   - 没有 json.loads 解析
#   - 没有 pydantic / schema 校验
#   - 没有重试逻辑
# 模型吐什么就打印什么。如果它返回一段散文，这个程序会毫无察觉、照样打印。
#
# 书里这段注释写的是 "Final JSON Output"，容易让人误以为已经得到结构化数据。
# 实际上「结构化」只发生在人的脑子里。
print("\n--- Final JSON Output ---")
print(final_result)



--- Final JSON Output ---
{
  "cpu": "3.5 GHz octa-core",
  "memory": "16GB",
  "storage": "1TB NVMe SSD"
}


## 执行时序：到底发生了什么

| # | 环节 | 类型 | 花 token | 输入 → 输出 |
|---|---|---|---|---|
| 1 | `prompt_extract` 插值 | 本地 | ❌ | `{"text_input": "..."}` → `ChatPromptValue` |
| 2 | `llm` 第 1 次调用 | **网络** | ✅ | `ChatPromptValue` → `AIMessage` |
| 3 | `StrOutputParser` | 本地 | ❌ | `AIMessage` → `TextAccessor`（str 子类） |
| 4 | dict 打包 | 本地 | ❌ | 字符串 → `{"specifications": "..."}` |
| 5 | `prompt_transform` 插值 | 本地 | ❌ | 同上 → `ChatPromptValue` |
| 6 | `llm` 第 2 次调用 | **网络** | ✅ | `ChatPromptValue` → `AIMessage` |
| 7 | `StrOutputParser` | 本地 | ❌ | `AIMessage` → `str` |

**两个反直觉的结论**：

1. **7 个环节里只有 2 个花 token。** 其余 5 个都是本地字符串/类型搬运。
   LangChain 的价值不在"让模型更聪明"，而在**把这 5 个搬运步骤标准化、可组合**。
2. **延迟 ≈ 2 × 单次模型调用。** 链每加一环（且含 LLM），延迟和成本线性叠加。

## 六个关键点（书里没明说，但决定你怎么用它）

### ① 惰性求值：`|` 不执行任何东西

```python
chain = a | b | c          # 零次调用
result = chain.invoke(x)   # 这时才开始跑
```

把"组装"和"执行"分开是 LCEL 的核心设计。好处：链可以复用、可以只跑前半截、
可以用 `.stream()` / `.batch()` 换执行方式而不用改组装逻辑。

### ② dict 就是「接口」，而且是个极脆弱的接口

```python
{"specifications": extraction_chain}
```

这一个 dict 承担了三件事：绑定变量名、声明数据流、隐式构造并行执行器。
但它**没有类型标注、没有 schema、没有校验**。上游吐什么，下游就信什么。

### ③ 全书最容易误解的一点：`final_result` 不是 JSON

它是 `str`。`print` 出来像 JSON，纯粹是模型配合表演。
如果模型返回：

```
Here are the specs: cpu is 3.5GHz...
```

这个脚本照样跑完、照样打印、照样"成功"。

### ④ 花括号是保留字符

想在一个 prompt 里写字面量 `{"cpu": "..."}`，必须写成 `{{"cpu": "..."}}`。
这就是为什么本书这个 prompt 用"with 'cpu', 'memory', and 'storage' as keys"
这种绕弯说法 —— 避开转义麻烦。

### ⑤ `temperature=0` ≠ 确定性

`deepseek-flash` 是带推理的模型（`usage_metadata` 里有 `reasoning` token 计数）。
推理路径本身带随机性，`temperature=0` 只压采样温度、冻不住推理。
实测中同一个输入曾给出 `"16GB"` 和 `"16 GB"` 两种结果。

### ⑥ 错误传播是单向且静默的

设第 2 步成功率为 p，链长为 n，整体近似 pⁿ。
p=0.95、n=2 → 0.90。**更糟的是错误不会被发现**：第 2 步无从知道第 1 步漏了信息。

举例：如果第 1 步漏掉了 "1TB NVMe SSD"，第 2 步**照样能编一个** `"storage": "1TB"`
出来 —— 格式完美、语气可信、内容错误。

> 这就是为什么后面必须要引入校验步：第 4 章 Reflection、第 12 章异常处理、
> 第 18 章 Guardrails。**Prompt chaining 提升的是可控性，不是正确性。**

In [2]:
# =============================================================================
# 验证：证明「看起来像 JSON」≠「是 JSON」
# =============================================================================
# 这一格不调用模型，只是检查上面那个 final_result 的真实身份。

print("type(final_result)  =", type(final_result).__name__)
print("是 str 吗           =", isinstance(final_result, str))
print("是 dict 吗          =", isinstance(final_result, dict))
print()

import json

try:
    parsed = json.loads(final_result)
    print("json.loads 成功 ->", type(parsed).__name__, "=", parsed)
except json.JSONDecodeError as e:
    print("json.loads 失败 ->", e)
    print("原始输出是：")
    print(final_result)
    print()
    print("结论：模型这次把 JSON 包在了 Markdown ``` 代码块里，")
    print("      所以必须先剥离围栏才能解析 —— 这正是『没有校验』的代价。")

type(final_result)  = TextAccessor
是 str 吗           = True
是 dict 吗          = False

json.loads 成功 -> dict = {'cpu': '3.5 GHz octa-core', 'memory': '16GB', 'storage': '1TB NVMe SSD'}


## 如果想把这段代码「生产化」，缺什么

按重要性排序：

| 缺什么 | 后果 | 正解 |
|---|---|---|
| 输出校验 | 模型返回散文也算"成功" | `json.loads` + pydantic schema（第 18 章） |
| 结构化输出 | 靠 prompt 祈祷格式正确 | `with_structured_output()` / JSON mode（第 5 章） |
| 失败重试 | 一次网络抖动全链作废 | 指数退避（第 12 章） |
| 可观测性 | 出错时不知道哪一步坏的 | tracing / LangSmith（第 19 章） |
| 中间态持久化 | 重跑要从头再来 | LangGraph checkpoint（第 8 章） |

**建议**：现在把上面 `verify` 格跑一遍，亲眼看一下 `final_result` 的真身。
如果 `json.loads` 失败，你就已经亲手体验了"链式系统静默出错"的第一个实例。